# Bronze profiling

First look at gzip CSV files under the bronze Hive layout:

```
data/bronze/locationid=<ID>/year=<YYYY>/location-<ID>-<YYYYMMDD>.csv.gz
```

In [1]:
from pathlib import Path

import pandas as pd

BRONZE_ROOT = Path("../data/bronze")
bronze_files = sorted(BRONZE_ROOT.rglob("*.csv.gz"))
assert bronze_files, f"No bronze files under {BRONZE_ROOT.resolve()}"

BRONZE_FILE = bronze_files[0]
df = pd.read_csv(BRONZE_FILE, compression="gzip")

print("=" * 80)
print("BRONZE OVERVIEW")
print("=" * 80)
print(f"\nSource: {BRONZE_FILE.relative_to(BRONZE_ROOT.parent)}")
print(f"Bronze files found: {len(bronze_files)}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Location: {df['location'].iloc[0]}")
print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"\nColumns: {list(df.columns)}")
df.head()

BRONZE OVERVIEW

Source: bronze\locationid=1544061\year=2026\location-1544061-20260101.csv.gz
Bronze files found: 7
Shape: 120 rows x 9 columns
Location: Anzac Memorial-1514036
Date range: 2026-01-01T01:00:00+11:00 → 2026-01-02T00:00:00+11:00

Columns: ['location_id', 'sensors_id', 'location', 'datetime', 'lat', 'lon', 'parameter', 'units', 'value']


,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value
0,1544061,6910832,Anzac Memorial-1514036,2026-01-01T01:00:00+11:00,-33.875728,151.138864,pm25,µg/m³,11.863095
1,1544061,6910832,Anzac Memorial-1514036,2026-01-01T02:00:00+11:00,-33.875728,151.138864,pm25,µg/m³,8.846726
2,1544061,6910832,Anzac Memorial-1514036,2026-01-01T03:00:00+11:00,-33.875728,151.138864,pm25,µg/m³,7.069940
3,1544061,6910832,Anzac Memorial-1514036,2026-01-01T04:00:00+11:00,-33.875728,151.138864,pm25,µg/m³,9.693452
4,1544061,6910832,Anzac Memorial-1514036,2026-01-01T05:00:00+11:00,-33.875728,151.138864,pm25,µg/m³,6.385417


## Schema and parameter inventory

In [2]:
print(df.dtypes)
print()

param_summary = (
    df.groupby("parameter")
    .agg(
        rows=("value", "count"),
        unit=("units", "first"),
        min=("value", "min"),
        max=("value", "max"),
        mean=("value", "mean"),
    )
    .sort_index()
)
param_summary

location_id      int64
sensors_id       int64
location           str
datetime           str
lat            float64
lon            float64
parameter          str
units              str
value          float64
dtype: object



,rows,unit,min,max,mean
parameter,,,,,
pm1,24,µg/m³,0.000000,6.834821,1.455831
pm25,24,µg/m³,0.151786,11.863095,3.157315
relativehumidity,24,%,44.438616,65.222530,55.363877
temperature,24,c,19.126845,21.151532,20.143918
um003,24,particles/cm³,221.366071,1515.663690,564.226207


## Layer 1 preview — E3 stuck values and E4 negatives

In [3]:
negatives = df[df["value"] < 0]
print(f"E4 — negative concentrations: {len(negatives)} rows")


def max_stuck_run(values: pd.Series) -> int:
    longest = run = 0
    prev = None
    for value in values:
        if value == prev:
            run += 1
            longest = max(longest, run)
        else:
            run = 1
        prev = value
    return longest


stuck = []
for param, group in df.groupby("parameter"):
    ordered = group.sort_values("datetime")["value"]
    stuck.append(
        {
            "parameter": param,
            "max_stuck_run": max_stuck_run(ordered),
            "variance": ordered.var(),
        }
    )

pd.DataFrame(stuck).sort_values("max_stuck_run", ascending=False)

E4 — negative concentrations: 0 rows


,parameter,max_stuck_run,variance
0,pm1,0,4.093052
1,pm25,0,10.527288
2,relativehumidity,0,55.065442
3,temperature,0,0.259538
4,um003,0,133414.193396
